## Workflow for Neosurf-on-Neosurf MaSIF search

In [31]:
import os
import pandas as pd
import sys

repo_root = !git rev-parse --show-toplevel
repo_root = repo_root[0]
os.chdir(repo_root)

# Add source_dir to python PATH
source_dir = os.path.join(repo_root, 'masif_seed_search/source')
sys.path.insert(0, source_dir)

# ----- Directories ------
data_dir = os.path.join(repo_root, 'data')

# ----- Input ------
# .csv about the seed complexes
seed_list_csv = os.path.join(data_dir, 'nico_targets.csv')

# Directory to store input .pdb and .sdf
input_dir = os.path.join(data_dir, 'input')
os.makedirs(input_dir, exist_ok=True)
input_manifest = os.path.join(input_dir, 'input_manifest.csv')

# ----- Output ------
# Directory to write preprocessing files
preprocess_dir = os.path.join(data_dir, 'preprocess')

# Directory to write masif-search output
masif_search_out_dir = os.path.join(data_dir, 'masif_search')
os.makedirs(masif_search_out_dir, exist_ok=True)


___
### Step 1 - preprocess all targets
1. Preprocess targets with ligands in nico_targets.csv
2. Preprocess VHL and CRBN with ligands

In [28]:
df_seed = pd.read_csv(seed_list_csv)

df_seed.head()

,uniprot_id,gene_name,recommendedName,pdb_id,protein_chain,ligand_chain,ligand_code,ligand_name,smiles,percent_intracellular,formula,mw,qed,num_carbon,num_N_O,uniprot_id_count
0,P35222,CTNNB1,Catenin beta-1,6M90,C,A,J91,2-(2-fluorophenoxy)-3-{[2-oxo-6-(trifluorometh...,c1ccc(c(c1)Oc2c(cccc2NC(=O)C3=CC=C(NC3=O)C(F)(...,1.0,C20H12F4N2O5,436.068234,0.517802,20.0,7.0,1.0
1,P35222,CTNNB1,Catenin beta-1,6M91,C,A,J97,"3-({4-[(2,6-dichlorophenyl)sulfanyl]-2-oxo-6-(...",c1cc(cc(c1)NC(=O)C2=C(C=C(NC2=O)C(F)(F)F)Sc3c(...,1.0,C20H11Cl2F3N2O4S,501.976868,0.408829,20.0,6.0,1.0
2,P35222,CTNNB1,Catenin beta-1,6M92,C,A,J8V,"3-{[2-oxo-4-phenoxy-6-(trifluoromethyl)-1,2-di...",c1ccc(cc1)OC2=C(C(=O)NC(=C2)C(F)(F)F)C(=O)Nc3c...,1.0,C20H13F3N2O5,418.077656,0.577285,20.0,7.0,1.0
3,P35222,CTNNB1,Catenin beta-1,6M93,C,A,J8Y,2-oxo-N-[3-(1H-tetrazol-5-yl)phenyl]-6-(triflu...,c1cc(cc(c1)NC(=O)C2=CC=C(NC2=O)C(F)(F)F)c3[nH]...,1.0,C14H9F3N6O2,350.073908,0.666230,14.0,8.0,1.0
4,P35222,CTNNB1,Catenin beta-1,7AFW,A,A,R9Q,"3-[(2~{R})-4-methyl-5-oxidanylidene-2,3-dihydr...",CN1C[C@H](Oc2ccccc2C1=O)c3cccc(c3)C#N,1.0,C17H14N2O2,278.105528,0.805631,17.0,4.0,1.0


In [29]:
# ----- Helper functions for preparing preprocessing inputs ------
#
# prepare_input_structures() orchestrates one row of df_seed (nico_targets.csv) into
# MaSIF-ready files under input_dir. Per complex the workflow is:
#
#   1. Download the full PDB from RCSB and the ligand SDF (models.rcsb.org) into
#      input_dir / a temp directory.
#   2. Trim the structure to the target protein (standard amino acids on protein_chain)
#      plus a single ligand residue (ligand_code on ligand_chain). If protein_chain
#      and ligand_chain are the same, both are kept on that chain.
#   3. Build a target name {pdb_id}_{chains}, where chains = protein_chain + ligand_chain
#      deduplicated (e.g. C+A -> CA, A+A -> A).
#   4. EvoEF2 RepairStructure on a protein-only PDB (ligand stripped); repaired
#      coordinates are written to a temp file as structure_Repair.pdb.
#   5. Merge the original ligand HETATM/ATOM records from the pre-repair trimmed complex
#      onto the repaired protein and save the final PDB to input_dir.
#   6. Return a dict for df_preprocess: pdb_path, target, ligand ({code}_{chain}),
#      ligand_path ({pdb_id}_{protein_chain}_{ligand_code}.sdf).
#
# Files are always rebuilt (no skip-if-exists). The driver cell loops df_seed and
# calls prepare_input_structures(row, input_dir, repo_root/EvoEF2/EvoEF2).

import shutil
import subprocess
import tempfile
from pathlib import Path
from urllib.error import HTTPError
from urllib.request import urlopen

from Bio.PDB import PDBIO, PDBParser, Select

# chains suffix: protein_chain + ligand_chain, deduplicated (C+A -> CA, A+A -> A)
STANDARD_AA = {
    "ALA", "ARG", "ASN", "ASP", "CYS", "GLN", "GLU", "GLY", "HIS", "ILE",
    "LEU", "LYS", "MET", "PHE", "PRO", "SER", "THR", "TRP", "TYR", "VAL",
    "MSE",
}

evoef2_bin = os.path.join(repo_root, "EvoEF2", "EvoEF2")

def chain_suffix(protein_chain, ligand_chain):
    if protein_chain == ligand_chain:
        return protein_chain
    return protein_chain + ligand_chain


def _is_standard_protein_residue(residue):
    return residue.id[0] == " " and residue.get_resname().strip() in STANDARD_AA


def _is_ligand_residue(residue, ligand_code):
    return residue.get_resname().strip() == ligand_code.strip()


class _ComplexSelect(Select):
    def __init__(self, protein_chain, ligand_chain, ligand_code):
        self.protein_chain = protein_chain
        self.ligand_chain = ligand_chain
        self.ligand_code = ligand_code
        self._chains = {protein_chain, ligand_chain}

    def accept_chain(self, chain):
        return chain.id in self._chains

    def accept_residue(self, residue):
        chain_id = residue.parent.id
        if chain_id == self.protein_chain:
            if self.protein_chain == self.ligand_chain:
                return _is_standard_protein_residue(residue) or _is_ligand_residue(
                    residue, self.ligand_code
                )
            return _is_standard_protein_residue(residue)
        if chain_id == self.ligand_chain:
            return _is_ligand_residue(residue, self.ligand_code)
        return False


class _ProteinOnlySelect(Select):
    def __init__(self, protein_chain):
        self.protein_chain = protein_chain

    def accept_chain(self, chain):
        return chain.id == self.protein_chain

    def accept_residue(self, residue):
        return _is_standard_protein_residue(residue)


def _download_url(url, dest_path):
    dest_path = Path(dest_path)
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with urlopen(url) as response:
            dest_path.write_bytes(response.read())
    except HTTPError as exc:
        raise RuntimeError(f"Download failed ({exc.code}): {url}") from exc


def download_pdb(pdb_id, dest_path):
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    _download_url(url, dest_path)


def download_ligand_sdf(pdb_id, ligand_chain, dest_path):
    url = (
        f"https://models.rcsb.org/v1/{pdb_id}/ligand"
        f"?label_asym_id={ligand_chain}&encoding=sdf"
    )
    _download_url(url, dest_path)


def _extract_ligand_pdb_lines(pdb_path, ligand_chain, ligand_code):
    lines = []
    with open(pdb_path) as handle:
        for line in handle:
            if not (line.startswith("HETATM") or line.startswith("ATOM")):
                continue
            if line[21] != ligand_chain:
                continue
            if line[17:20].strip() != ligand_code.strip():
                continue
            lines.append(line)
    if not lines:
        raise ValueError(
            f"No ligand records for resname={ligand_code} chain={ligand_chain} in {pdb_path}"
        )
    return lines


def _merge_repaired_protein_with_ligand(repaired_pdb, trimmed_pdb, ligand_chain, ligand_code, output_pdb):
    ligand_lines = _extract_ligand_pdb_lines(trimmed_pdb, ligand_chain, ligand_code)
    out_lines = []
    with open(repaired_pdb) as handle:
        for line in handle:
            if line.startswith("END"):
                break
            if line.strip():
                out_lines.append(line)
    out_lines.extend(ligand_lines)
    out_lines.append("END\n")
    Path(output_pdb).write_text("".join(out_lines))


def evoef2_repair_structure(input_pdb, output_pdb, evoef2_bin):
    """Run EvoEF2 RepairStructure; intermediate {stem}_Repair.pdb lives in a temp directory."""
    evoef2_bin = str(evoef2_bin)
    if not os.path.isfile(evoef2_bin):
        raise FileNotFoundError(
            f"EvoEF2 binary not found at {evoef2_bin}. Build with: cd EvoEF2 && ./build.sh"
        )
    output_pdb = Path(output_pdb)
    output_pdb.parent.mkdir(parents=True, exist_ok=True)

    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        stem = "structure"
        work_pdb = tmp / f"{stem}.pdb"
        shutil.copy2(input_pdb, work_pdb)
        subprocess.run(
            [
                evoef2_bin,
                "--command=RepairStructure",
                f"--pdb={stem}.pdb"
            ],
            cwd=tmp,
            check=True,
        )
        repaired = tmp / f"{stem}_Repair.pdb"
        if not repaired.is_file():
            raise FileNotFoundError(f"EvoEF2 did not create {repaired}")
        shutil.copy2(repaired, output_pdb)


def prepare_input_structures(pdb_id, protein_chain, ligand_chain, ligand_code, input_dir, evoef2_bin):
    """
    Download PDB/SDF, trim to protein+ligand, EvoEF2-repair protein, merge ligand, write to input_dir.

    chains suffix: protein_chain + ligand_chain (deduplicated if equal).
    """
    input_dir = Path(input_dir)

    chains = chain_suffix(protein_chain, ligand_chain)
    target = f"{pdb_id}_{chains}"
    ligand_name = f"{ligand_code}_{ligand_chain}"
    pdb_path = input_dir / f"{target}.pdb"
    ligand_path = input_dir / f"{pdb_id}_{protein_chain}_{ligand_code}.sdf"

    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        full_pdb = tmp / f"{pdb_id}.pdb"
        download_pdb(pdb_id, full_pdb)
        download_ligand_sdf(pdb_id, ligand_chain, ligand_path)

        parser = PDBParser(QUIET=True)
        structure = parser.get_structure(pdb_id, str(full_pdb))

        trimmed_pdb = tmp / "trimmed.pdb"
        pdb_io = PDBIO()
        pdb_io.set_structure(structure)
        pdb_io.save(str(trimmed_pdb), _ComplexSelect(protein_chain, ligand_chain, ligand_code))
        _extract_ligand_pdb_lines(trimmed_pdb, ligand_chain, ligand_code)

        protein_only_pdb = tmp / "protein_only.pdb"
        pdb_io.set_structure(structure)
        pdb_io.save(str(protein_only_pdb), _ProteinOnlySelect(protein_chain))

        repaired_protein_pdb = tmp / "repaired_protein.pdb"
        evoef2_repair_structure(
            protein_only_pdb, repaired_protein_pdb, evoef2_bin
        )
        _merge_repaired_protein_with_ligand(
            repaired_protein_pdb, trimmed_pdb, ligand_chain, ligand_code, pdb_path
        )

    return {
        "pdb_path": str(pdb_path.resolve()),
        "target": target,
        "ligand": ligand_name,
        "ligand_path": str(ligand_path.resolve()),
    }



In [ ]:
# Iterate over all rows in df_seed
rows = []
for _, row in df_seed.iterrows():
    print(f"Preparing {row['pdb_id']} ({row['protein_chain']} + {row['ligand_chain']})...")
    pdb_id = row["pdb_id"]
    protein_chain = row["protein_chain"]
    ligand_chain = row["ligand_chain"]
    ligand_code = row["ligand_code"]
    rows.append(prepare_input_structures(pdb_id, protein_chain, ligand_chain, ligand_code, input_dir, evoef2_bin))

df_preprocess = pd.DataFrame(rows)
df_preprocess.head()

In [ ]:
# Add VHL structure as well:
pdb_id = "8VLB"
protein_chain = "A"
ligand_chain = "A"
ligand_code = "3JF"


rows.append(prepare_input_structures(pdb_id, protein_chain, ligand_chain, ligand_code, input_dir, evoef2_bin))

df_preprocess = pd.DataFrame(rows)

In [ ]:
# Add CRBN-pomalidomide structure as well:
pdb_id = "6H0F"
protein_chain = "B"
ligand_chain = "B"
ligand_code = "Y70"

rows.append(prepare_input_structures(pdb_id, protein_chain, ligand_chain, ligand_code, input_dir, evoef2_bin))

df_preprocess = pd.DataFrame(rows)

In [ ]:
# Write df_preprocess to input_manifest.csv, then run preprocess_array.sh
df_preprocess.to_csv(input_manifest, index=False)
n_rows = len(df_preprocess)
manifest_abs = os.path.abspath(input_manifest)
# Use {var} for Python values in ! commands — ${var} becomes a literal '$' + path in Jupyter
!sbatch --array=1-{n_rows} scripts/slurm/preprocess_array.sh {manifest_abs}

___
### Step 2 - Run masif_search around ligand residues
1. Query with VHL structure as target, search for seed patches around the seed ligands

In [ ]:
from pathlib import Path

df_preprocess = pd.read_csv(input_manifest)

QUERY_TARGET = "8VLB_A"
query_out_dir = os.path.join(masif_search_out_dir, QUERY_TARGET)
subset_dir = Path(query_out_dir) / "subset"

# Clear the subset directory before writing new seeds
if subset_dir.exists():
    shutil.rmtree(subset_dir)
subset_dir.mkdir(parents=True, exist_ok=True)

# Seeds to search (all preprocessed targets except the query)
seed_ids = df_preprocess.loc[df_preprocess["target"] != QUERY_TARGET, "target"].tolist()

# Control: use seeds from 8VLB_domainome_sampled_hits.txt
# seed_ids = pd.read_csv("/scratch/ymeng/NNeosurf/masif-neosurf/data/8VLB_domainome_sampled_hits.txt", header=None).iloc[:, 0].tolist()

# Evenly split all seed ids across N_SUBSET files (one masif_search.py call per file)
N_SUBSET = 500
chunks = [[] for _ in range(N_SUBSET)]
for i, seed_id in enumerate(seed_ids):
    chunks[i % N_SUBSET].append(seed_id)

for chunk_ix, chunk in enumerate(chunks, start=1):
    if chunk:
        (subset_dir / str(chunk_ix)).write_text("\n".join(chunk) + "\n")

n_subsets = sum(1 for chunk in chunks if chunk)

subset_dir_abs = os.path.abspath(subset_dir)
print(f"Query target: {QUERY_TARGET}")
print(f"Wrote {len(seed_ids)} seed(s) into {n_subsets} subset file(s) under {subset_dir}")
print(
    f"Submit: sbatch --array=1-{n_subsets} scripts/slurm/search_array.sh "
    f"{QUERY_TARGET} {query_out_dir} {subset_dir_abs}"
)


Query target: 8VLB_A
Wrote 5000 seed(s) into 500 subset file(s) under /scratch/ymeng/NNeosurf/masif-neosurf/data/masif_search/8VLB_A/subset
Submit: sbatch --array=1-500 scripts/slurm/search_array.sh 8VLB_A /scratch/ymeng/NNeosurf/masif-neosurf/data/masif_search/8VLB_A /scratch/ymeng/NNeosurf/masif-neosurf/data/masif_search/8VLB_A/subset


In [ ]:
query_out_dir_abs = os.path.abspath(query_out_dir)
!sbatch --array=1-{n_subsets} scripts/slurm/search_array.sh {QUERY_TARGET} {query_out_dir_abs} {subset_dir_abs}

In [38]:
QUERY_TARGET = "6H0F_B"

In [39]:
# Read df_results from clustered_matches/*.csv
clustered_match_dir = os.path.join(query_out_dir, QUERY_TARGET, "clustered_matches")
results_csv = os.path.join(query_out_dir, f"{QUERY_TARGET}_search_results.csv")

df_results = pd.DataFrame()
for match_file in os.listdir(clustered_match_dir):
    if match_file.endswith(".csv"):
        df_match = pd.read_csv(os.path.join(clustered_match_dir, match_file))
        df_results = pd.concat([df_results, df_match])

df_results.to_csv(results_csv, index=False)
print(f"Wrote {len(df_results)} matches to {results_csv}")
df_results.head()

FileNotFoundError: [Errno 2] No such file or directory: '/scratch/ymeng/NNeosurf/masif-neosurf/data/masif_search/8VLB_A/6H0F_B/clustered_matches'

In [35]:
# Slice max to each unique combination of target, matched_protein, cluster_id with the highest score
df_dedup = df_results.sort_values(by="score", ascending=False).groupby(["target", "matched_protein", "cluster_id"]).first().reset_index()
dedup_csv = os.path.join(query_out_dir, f"{QUERY_TARGET}_dedup.csv")

df_dedup.to_csv(dedup_csv, index=False)
print(f"Wrote {len(df_dedup)} deduplicated matches to {dedup_csv}")
df_dedup.head()


Wrote 16 deduplicated matches to /scratch/ymeng/NNeosurf/masif-neosurf/data/masif_search/8VLB_A/8VLB_A_dedup.csv


,target,matched_protein,cluster_id,target_site,target_vix,matched_patch_id,score,desc_dist_score,clashing_ca,clashing_heavy,matched_vix,desc_dist,iface_score,mean_desc_dist_score,flattened_transform,cluster_size,cluster_mean_rmsd
0,8VLB_A,5QSN_B,0.0,site_29,2436,16,0.929423,14.487843,0,1,971,2.862443,0.295589,0.139306,"0.65886559964854,-0.5842780612126817,0.4738304...",1.0,0.000000
1,8VLB_A,5QSO_A,0.0,site_13,1151,83,0.986322,18.636735,0,2,7999,2.464726,0.251634,0.137035,"0.19343810258248714,-0.6022838269667254,0.7744...",25.0,1.063955
2,8VLB_A,5QSO_A,1.0,site_38,2697,75,0.980178,14.308082,0,1,8214,2.463412,0.407524,0.118249,"-0.3627953640377072,0.27992761312418757,0.8888...",3.0,1.832128
3,8VLB_A,5QSV_D,0.0,site_22,2799,17,0.949333,12.216403,0,0,1325,3.171358,0.285579,0.104414,"0.027919075278252103,0.10099289635807518,0.994...",1.0,0.000000
4,8VLB_A,5QSV_D,1.0,site_34,2179,98,0.943282,20.194905,0,2,5576,1.900894,0.559067,0.160277,"0.886549788328148,0.4054343225799812,-0.222828...",1.0,0.000000


In [37]:
def print_pymol_commands(row):
    matched_protein = row["matched_protein"]
    matched_pdb = matched_protein.split("_")[0]
    matched_chains = matched_protein.split("_")[1]
    matched_patch_id = row["matched_patch_id"]
    flattened_transform = row["flattened_transform"]

    # Split matched_chains (e.g. "AB") into ["A", "B"]
    matched_chains = list(matched_chains)
    # Joint matched_chains into a string (e.g. "chain A chain B")
    matched_chains_str = "chain " + " chain ".join(matched_chains)

    print(f"fetch {matched_pdb}, {matched_protein}_{matched_patch_id}")
    #print(f"select {matched_protein}_{matched_patch_id} AND (not {matched_chains_str})")
    #print('cmd.remove("sele");cmd.delete("sele")')
    print(f"apply_transform {matched_protein}_{matched_patch_id}, '{flattened_transform}'")

# Apply to all rows
df_dedup.apply(print_pymol_commands, axis=1)

fetch 5QSN, 5QSN_B_16
apply_transform 5QSN_B_16, '0.65886559964854,-0.5842780612126817,0.4738304219711079,101.69230588110312,-0.5306812573973989,0.08542589964307447,0.8432554884003337,43.35899458289698,-0.5331730719446401,-0.8070449571465771,-0.25378122960020955,87.49713313389961,0.0,0.0,0.0,1.0'
fetch 5QSO, 5QSO_A_83
apply_transform 5QSO_A_83, '0.19343810258248714,-0.6022838269667254,0.7744907308958595,118.80644490498624,0.2441548683602201,-0.7350156557638269,-0.6325665072054277,50.61712922695046,0.9502473892230857,0.3114581473591383,0.00487049466917322,92.87160382814618,0.0,0.0,0.0,1.0'
fetch 5QSO, 5QSO_A_75
apply_transform 5QSO_A_75, '-0.3627953640377072,0.27992761312418757,0.8888307236157755,61.69024896200561,0.030542069062689498,-0.9497309598667882,0.31157388511856954,79.4393795704846,0.9313681902717732,0.1401842904219475,0.33600842083015947,106.45533129949402,0.0,0.0,0.0,1.0'
fetch 5QSV, 5QSV_D_17
apply_transform 5QSV_D_17, '0.027919075278252103,0.10099289635807518,0.994495329360

0     None
1     None
2     None
3     None
4     None
5     None
6     None
7     None
8     None
9     None
10    None
11    None
12    None
13    None
14    None
15    None
dtype: object